# Item-to-item: LLM reranking with user profile (oracle-ish upper bound)

**Goal:** keep the same **co-preference item-to-item** evaluation as other processed notebooks, but replace (or augment) pure embedding nearest-neighbors with a **two-stage** pipeline:

1) **dense retrieval** builds a candidate pool (fast)
2) an **LLM** reranks candidates using **USER_PROFILE** + **QUERY_DISH** text (slow, expensive)

**Fair comparison (important):** we report both (a) a legacy **full-catalog** dense baseline and (b) a **top-pool-only** dense baseline on the *same* `CANDIDATE_POOL` neighbors and the *same* evaluated users as the LLM reranker. Comparing (b) vs LLM answers “does reranking help inside a fixed candidate pool?”.

This notebook is intentionally "last mile": it uses extra supervision signals that a production system might not have at scoring time, but it answers the practical question: **how much headroom** is left if the ranker can read the profile + dish facts.

## What the metric is (and why it is tiny)

We measure **retrieval quality** under a sparse label definition:

- Take test users with enough positives (`order`/`favorite`).
- Pick one positive dish as **query**.
- Treat the remaining positives as **relevant** items.
- Retrieve from the full catalog and compute IR metrics (`P@K`, `NDCG@K`, `MRR`, ...).

**Why `P@10` is naturally near-zero:** even if the model is good, you are asking for "the needle" among thousands of dishes using **very few positives** as labels. `P@10` divides hits by **10**, not by the number of relevant items, so the absolute scale stays small unless retrieval is extremely peaked.

**Why it is noisy:** older notebooks used `query_dish = list(set_positives)[0]`; set iteration order is not a stable "random choice" across machines. Here we default to a **deterministic** query (`sorted positives[0]`) to reduce pointless variance between runs.

## Prereqs

- GPU recommended for encoding.
- LLM access via OpenAI-compatible API (OpenRouter by default), using the same env conventions as `scripts/augment_dishes_retrieval_llm.py` (`OPENAI_API_KEY` / `OPENAI_BASE_URL`, or `ORACLE_*` in `backend/.env`, or repo root `.env`).
- **Kaggle:** create secrets `OPENAI_API_KEY` (required) and usually `OPENAI_BASE_URL` (e.g. `https://openrouter.ai/api/v1`), then run the **“KAGGLE SECRETS → os.environ”** cell below. Also enable **Internet** in notebook settings if your API is remote.


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU recommended for encoding (enable GPU on Colab/Kaggle)."
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content' if ENV == 'colab' else os.getcwd()
REPO_DIR = f'{BASE}/{REPO}' if ENV != 'local' else os.path.abspath(os.path.join(os.getcwd(), '..'))

if ENV != 'local':
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow tqdm openai')
sys.path.insert(0, f'{REPO_DIR}/src')

print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")


In [ ]:
# ============================================================
# DATA PATHS
# ============================================================
import os

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'

if ENV == 'local':
    SYNTHETIC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'synthetic')
else:
    SYNTHETIC_DIR = os.path.join(REPO_DIR, 'data', 'synthetic')

OUTPUT_DIR = '/kaggle/working/processed' if ENV == 'kaggle' else SYNTHETIC_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"SYNTHETIC_DIR = {SYNTHETIC_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")


In [ ]:
# ============================================================
# KAGGLE SECRETS → os.environ (run BEFORE LLMProfileReranker)
# ============================================================
import os

if os.path.exists("/kaggle/working"):
    try:
        from kaggle_secrets import UserSecretsClient

        sec = UserSecretsClient()
        # Create these secret names in Kaggle UI (Add-ons → Secrets)
        for name in (
            "OPENAI_API_KEY",
            "OPENAI_BASE_URL",
            "ORACLE_API_KEY",
            "ORACLE_API_BASE",
        ):
            try:
                val = sec.get_secret(name)
            except Exception:
                continue
            if val:
                os.environ[name] = val

        print(
            "Kaggle secrets loaded into env:",
            {k: bool(os.environ.get(k)) for k in ("OPENAI_API_KEY", "OPENAI_BASE_URL", "ORACLE_API_KEY", "ORACLE_API_BASE")},
        )
    except Exception as e:
        print(f"Kaggle secrets not available ({e}). Set env vars manually if needed.")
else:
    print("Not on Kaggle — use local .env / notebook env vars as usual.")


In [ ]:
# ============================================================
# PARAMETERS
# ============================================================

# Stage-1 pool size (dense neighbors before LLM rerank)
CANDIDATE_POOL = 50

# Cost control: LLM reranking is O(users). Start small.
MAX_USERS_FOR_LLM = int(os.environ.get("MAX_USERS_FOR_LLM", "120"))

# Query selection:
# - "sorted_first": deterministic (recommended)
# - "legacy_set_first": matches older notebooks (can be unstable across runs)
QUERY_MODE = os.environ.get("QUERY_MODE", "sorted_first")

LLM_MODEL = os.environ.get("LLM_RERANK_MODEL", "openai/gpt-4o-mini")

# Fair comparison vs LLM rerank:
# compute a dense baseline on the SAME top-CANDIDATE_POOL neighbor list and the SAME user subset.
INCLUDE_DENSE_POOL_BASELINE = os.environ.get("INCLUDE_DENSE_POOL_BASELINE", "1") not in ("0", "false", "False")

# Optional: keep the legacy full-catalog dense baseline (can dominate recall vs pool-only methods).
INCLUDE_DENSE_FULL_BASELINE = os.environ.get("INCLUDE_DENSE_FULL_BASELINE", "1") not in ("0", "false", "False")

# LLM HTTP robustness / prompt size (see src/llm_profile_reranker.py)
# - LLM_RERANK_TIMEOUT_S: per-request timeout (seconds)
# - LLM_RERANK_MAX_RETRIES: attempts including JSON-parse retries
# - LLM_RERANK_CAND_TEXT_CHARS: truncate each candidate line in the prompt
# - LLM_RERANK_DEBUG=1: print per-request attempt logs
_llm_timeout = os.environ.get("LLM_RERANK_TIMEOUT_S", "60")
_llm_retries = os.environ.get("LLM_RERANK_MAX_RETRIES", "4")
_llm_cand_chars = os.environ.get("LLM_RERANK_CAND_TEXT_CHARS", "280")
_llm_debug = os.environ.get("LLM_RERANK_DEBUG", "0")

print({
    "CANDIDATE_POOL": CANDIDATE_POOL,
    "MAX_USERS_FOR_LLM": MAX_USERS_FOR_LLM,
    "QUERY_MODE": QUERY_MODE,
    "LLM_MODEL": LLM_MODEL,
    "INCLUDE_DENSE_POOL_BASELINE": INCLUDE_DENSE_POOL_BASELINE,
    "INCLUDE_DENSE_FULL_BASELINE": INCLUDE_DENSE_FULL_BASELINE,
    "LLM_RERANK_TIMEOUT_S": _llm_timeout,
    "LLM_RERANK_MAX_RETRIES": _llm_retries,
    "LLM_RERANK_CAND_TEXT_CHARS": _llm_cand_chars,
    "LLM_RERANK_DEBUG": _llm_debug,
})


In [ ]:
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm

from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from utils import evaluate_all
from llm_profile_reranker import LLMProfileReranker, user_row_to_profile_text

np.random.seed(42)

dishes = pd.read_parquet(f"{SYNTHETIC_DIR}/dishes.parquet")
users = pd.read_parquet(f"{SYNTHETIC_DIR}/users.parquet")
test = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_test.parquet")

dish_id_to_idx = {int(did): i for i, did in enumerate(dishes['id'].tolist())}
idx_to_dish_id = {i: int(did) for i, did in enumerate(dishes['id'].tolist())}

dish_rows = {int(r['id']): r for _, r in dishes.iterrows()}

FLAG_KW = dict(
    include_recipe=False,
    include_macro_tokens=False,
    include_ratios=True,
    include_ingredients=True,
)


def dish_text(dish_id: int) -> str:
    row = dish_rows[int(dish_id)]
    tags = row.get('tag_list', [])
    return dish_to_rich_text(row.to_dict(), tags=tags, **FLAG_KW)


texts = [dish_text(int(did)) for did in dishes['id'].tolist()]

model = EmbeddingModel()
print(f"Model: {model.model_name} | dim={model.dim}")

embs = model.encode(texts, batch_size=64).astype(np.float32)

index = faiss.IndexFlatIP(model.dim)
index.add(embs)

print("FAISS index ready")


In [ ]:
user_positives = (
    test[test['interaction_type'].isin(['order', 'favorite'])]
    .groupby('user_id')['dish_id']
    .apply(set)
    .to_dict()
)

user_id_to_profile = {int(r['user_id']): user_row_to_profile_text(r.to_dict()) for _, r in users.iterrows()}

reranker = LLMProfileReranker(model=LLM_MODEL)
print({"llm_available": bool(reranker.available), "llm_model": LLM_MODEL})


def pick_query_dish(pos: set[int]) -> int:
    pos_in = sorted({int(d) for d in pos if int(d) in dish_id_to_idx})
    if not pos_in:
        raise ValueError("empty positives")
    if QUERY_MODE == "legacy_set_first":
        return int(list(set(pos_in))[0])
    # default: deterministic
    return int(pos_in[0])


def evaluate_item_to_item_dense_full(index, embeddings, ks=None):
    """Legacy baseline: dense neighbors with k_search = max(ks)+1 (sees beyond a fixed pool)."""
    if ks is None:
        ks = [5, 10, 20]
    k_search = max(ks) + 1

    rows = []
    for uid, pos in user_positives.items():
        pos_dishes = {int(d) for d in pos if int(d) in dish_id_to_idx}
        if len(pos_dishes) < 5:
            continue

        query_dish = pick_query_dish(pos_dishes)
        relevant = pos_dishes - {query_dish}

        q_idx = dish_id_to_idx[query_dish]
        _, inds = index.search(embeddings[q_idx : q_idx + 1], k_search)
        recommended = [
            idx_to_dish_id[i] for i in inds[0] if i >= 0 and idx_to_dish_id.get(i) != query_dish
        ]
        rows.append(evaluate_all(recommended, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


def _llm_eval_user_ids(max_users: int) -> list[int]:
    eval_users: list[int] = []
    for uid, pos in user_positives.items():
        pos_dishes = {int(d) for d in pos if int(d) in dish_id_to_idx}
        if len(pos_dishes) < 5:
            continue
        if int(uid) not in user_id_to_profile:
            continue
        eval_users.append(int(uid))

    return eval_users[: int(max_users)]


def evaluate_item_to_item_dense_pool(
    index,
    embeddings,
    ks=None,
    max_users: int = 200,
    pool: int = 50,
):
    """Fair dense baseline: rank is restricted to top-`pool` FAISS neighbors (excluding query)."""
    if ks is None:
        ks = [5, 10, 20]

    k_search = max(int(pool), max(ks)) + 5
    eval_users = _llm_eval_user_ids(max_users)

    rows = []
    for uid in tqdm(eval_users, desc="Dense pool eval users"):
        pos = user_positives[uid]
        pos_dishes = {int(d) for d in pos if int(d) in dish_id_to_idx}
        query_dish = pick_query_dish(pos_dishes)
        relevant = pos_dishes - {query_dish}

        q_idx = dish_id_to_idx[query_dish]
        _, inds = index.search(embeddings[q_idx : q_idx + 1], k_search)

        cand_ids: list[int] = []
        for i in inds[0].tolist():
            if i < 0:
                continue
            did = int(idx_to_dish_id[i])
            if did == int(query_dish):
                continue
            cand_ids.append(did)
            if len(cand_ids) >= int(pool):
                break

        rows.append(evaluate_all(cand_ids, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


def evaluate_item_to_item_llm_rerank(index, embeddings, ks=None, max_users: int = 200, pool: int = 50):
    if ks is None:
        ks = [5, 10, 20]

    k_search = max(pool, max(ks)) + 5
    eval_users = _llm_eval_user_ids(max_users)

    rows = []
    for uid in tqdm(eval_users, desc="LLM rerank eval users"):
        pos = user_positives[uid]
        pos_dishes = {int(d) for d in pos if int(d) in dish_id_to_idx}
        query_dish = pick_query_dish(pos_dishes)
        relevant = pos_dishes - {query_dish}

        profile_text = user_id_to_profile[int(uid)]
        q_idx = dish_id_to_idx[query_dish]
        _, inds = index.search(embeddings[q_idx : q_idx + 1], k_search)

        cand_ids = []
        for i in inds[0].tolist():
            if i < 0:
                continue
            did = int(idx_to_dish_id[i])
            if did == int(query_dish):
                continue
            cand_ids.append(did)
            if len(cand_ids) >= int(pool):
                break

        cands = [(did, dish_text(did)) for did in cand_ids]

        if reranker.available:
            ranked = reranker.rerank(
                profile_text=profile_text,
                query_dish_text=dish_text(int(query_dish)),
                candidates=cands,
                top_k=max(ks),
            )
            tail = [d for d in cand_ids if d not in set(ranked)]
            recommended = ranked + tail
        else:
            recommended = cand_ids

        rows.append(evaluate_all(recommended, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


metrics = {}

if INCLUDE_DENSE_FULL_BASELINE:
    metrics["dense_full_catalog"] = evaluate_item_to_item_dense_full(index, embs)

if INCLUDE_DENSE_POOL_BASELINE:
    metrics["dense_top_pool_only"] = evaluate_item_to_item_dense_pool(
        index,
        embs,
        max_users=MAX_USERS_FOR_LLM,
        pool=CANDIDATE_POOL,
    )

metrics["dense_top_pool_llm_rerank"] = evaluate_item_to_item_llm_rerank(
    index,
    embs,
    max_users=MAX_USERS_FOR_LLM,
    pool=CANDIDATE_POOL,
)

cmp = pd.DataFrame(metrics).T
print(cmp[["P@5", "P@10", "NDCG@10", "MRR"]].round(4))

if INCLUDE_DENSE_POOL_BASELINE:
    m_pool = metrics.get("dense_top_pool_only", {})
    m_llm = metrics.get("dense_top_pool_llm_rerank", {})
    print(f"\nΔ P@10 (llm - dense_pool): {m_llm.get('P@10', 0) - m_pool.get('P@10', 0):+.4f}")

if INCLUDE_DENSE_FULL_BASELINE and INCLUDE_DENSE_POOL_BASELINE:
    m_full = metrics.get("dense_full_catalog", {})
    m_pool = metrics.get("dense_top_pool_only", {})
    print(f"\nΔ P@10 (dense_pool - dense_full): {m_pool.get('P@10', 0) - m_full.get('P@10', 0):+.4f}")


In [ ]:
import json
import os

out = {
    "metrics": metrics,
    "comparison": cmp.reset_index().rename(columns={"index": "method"}).to_dict(orient="records"),
    "params": {
        "CANDIDATE_POOL": CANDIDATE_POOL,
        "MAX_USERS_FOR_LLM": MAX_USERS_FOR_LLM,
        "QUERY_MODE": QUERY_MODE,
        "LLM_MODEL": LLM_MODEL,
        "llm_available": bool(reranker.available),
        "INCLUDE_DENSE_FULL_BASELINE": INCLUDE_DENSE_FULL_BASELINE,
        "INCLUDE_DENSE_POOL_BASELINE": INCLUDE_DENSE_POOL_BASELINE,
    },
}

out_path = os.path.join(OUTPUT_DIR, "results_item_to_item_llm_profile_rerank.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)

print(f"Saved: {out_path}")
